# **Installation**

In [1]:
!pip install -q lightgbm catboost imbalanced-learn xgboost scikit-learn pandas numpy
!pip install ipython-autotime

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 10.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 34.3 MB/s eta 0:00:00


# **Import Libraries**

In [2]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')
from scipy.stats import uniform

from sklearn.model_selection import train_test_split, StratifiedKFold, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (make_scorer, accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, matthews_corrcoef, cohen_kappa_score, confusion_matrix, balanced_accuracy_score)
from imblearn.metrics import geometric_mean_score
from sklearn.model_selection import KFold
from imblearn.over_sampling import ADASYN
import time
from scipy.stats import uniform

# **Load Dataset**

In [3]:
df = pd.read_csv('/content/bank-additional-full.csv', sep=';')

# **Data Preprocessing**

In [4]:
# Drop Leakage Feature
df.drop('duration', axis=1, inplace=True)

# Handle "unknown" Values
for col in df.select_dtypes(include='object').columns:
    df[col] = df[col].replace('unknown', df[col].mode()[0])

# Encode Target & One-Hot Encoding
df['y'] = df['y'].map({'no': 0, 'yes': 1})
df = pd.get_dummies(df, drop_first=True)

# Split Features & Target
X = df.drop('y', axis=1)
y = df['y']

# Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# **Scaling**


In [5]:
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)


# **Sampling Technique**

In [6]:
adasyn = ADASYN(random_state=42)
X_train_ad, y_train_ad = adasyn.fit_resample(X_train_s, y_train)

# **Cross Validation Setup**

In [7]:
skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

# **Evaluation Function**

In [8]:
results = []
def run_model_evaluation(model_name, clf_model):
    global results
    fold_wise_data = []


    for i, (train_idx, test_idx) in enumerate(skf.split(X_train_ad, y_train_ad)):

        X_tr, X_tes = X_train_ad[train_idx], X_train_ad[test_idx]
        y_tr, y_tes = y_train_ad[train_idx], y_train_ad[test_idx]

        # Training start
        t_start = time.time()
        clf_model.fit(X_tr, y_tr)
        t_end = time.time()

        # Predictions
        preds = clf_model.predict(X_tes)
        probs = clf_model.predict_proba(X_tes)[:, 1]


        tn, fp, fn, tp = confusion_matrix(y_tes, preds).ravel()
        re = tp / (tp + fn)
        spec = tn / (tn + fp)

        # Dictionary for each fold
        metrics = {
            "Fold": i + 1,
            "Classifier": model_name,
            "Accuracy": round(accuracy_score(y_tes, preds), 4),
            "Precision": round(precision_score(y_tes, preds, zero_division=0), 4),
            "Recall": round(re, 4),
            "Specificity": round(spec, 4),
            "G-Mean": round(np.sqrt(re * spec), 4),
            "F1-Score": round(f1_score(y_tes, preds), 4),
            "MCC": round(matthews_corrcoef(y_tes, preds), 4),
            "Kappa": round(cohen_kappa_score(y_tes, preds), 4),
            "ROC-AUC": round(roc_auc_score(y_tes, probs), 4),
            "Balanced Accuracy": round(balanced_accuracy_score(y_tes, preds), 4),
            "Train_Time": round(t_end - t_start, 4)
        }
        fold_wise_data.append(metrics)

    # Result
    df_fold = pd.DataFrame(fold_wise_data)
    print(f"--- For: {model_name} ---")
    display(df_fold)


    results.extend(fold_wise_data)



# **LogisticRegression**

In [11]:
from sklearn.linear_model import LogisticRegression
#  Model base initialize
model = LogisticRegression(max_iter=2000, random_state=42)

# Parameter distribution setup
params = {
    'C': [0.001, 0.01, 0.1, 1, 10, 100],
    'solver': ['liblinear', 'lbfgs'],
    'penalty': ['l2']
}

#  Randomized Search execution
rs = RandomizedSearchCV(
    estimator=model,
    param_distributions=params,
    n_iter=10,
    cv=skf,
    scoring='f1',
    n_jobs=-1,
    random_state=42
)

# Training
rs.fit(X_train_ad, y_train_ad)

# Best model extract
final_lr_model = rs.best_estimator_

print(f"Best Parameters: {rs.best_params_}")
print(f"Best CV Accuracy: {round(rs.best_score_, 4)}")

run_model_evaluation("Logistic Regression", final_lr_model)

Best Parameters: {'solver': 'lbfgs', 'penalty': 'l2', 'C': 0.1}
Best CV Accuracy: 0.6516
--- For: Logistic Regression ---


,Fold,Classifier,Accuracy,Precision,Recall,Specificity,G-Mean,F1-Score,MCC,Kappa,ROC-AUC,Balanced Accuracy,Train_Time
0,1,Logistic Regression,0.6950,0.7477,0.5879,0.8020,0.6866,0.6582,0.3991,0.3899,0.7533,0.6949,0.6208
1,2,Logistic Regression,0.6907,0.7362,0.5937,0.7876,0.6838,0.6573,0.3887,0.3814,0.7510,0.6907,0.7052
2,3,Logistic Regression,0.6806,0.7265,0.5781,0.7828,0.6727,0.6439,0.3688,0.3610,0.7387,0.6805,0.6864
3,4,Logistic Regression,0.6861,0.7365,0.5785,0.7934,0.6775,0.6480,0.3808,0.3720,0.7457,0.6860,0.6486
4,5,Logistic Regression,0.6857,0.7346,0.5805,0.7907,0.6775,0.6485,0.3797,0.3713,0.7456,0.6856,0.6428
5,6,Logistic Regression,0.6914,0.7375,0.5932,0.7893,0.6843,0.6575,0.3902,0.3826,0.7498,0.6913,0.6310
6,7,Logistic Regression,0.6984,0.7549,0.5867,0.8098,0.6893,0.6602,0.4068,0.3966,0.7553,0.6983,0.5662
7,8,Logistic Regression,0.6779,0.7242,0.5733,0.7821,0.6697,0.6400,0.3635,0.3556,0.7371,0.6777,0.4981
8,9,Logistic Regression,0.6960,0.7486,0.5896,0.8023,0.6877,0.6596,0.4010,0.3919,0.7467,0.6959,0.6129
9,10,Logistic Regression,0.6833,0.7370,0.5694,0.7971,0.6737,0.6424,0.3764,0.3666,0.7445,0.6832,0.5863


# **Decision Tree**

In [12]:
from sklearn.tree import DecisionTreeClassifier
#  Model base initialize
model = DecisionTreeClassifier(random_state=42)

# Parameter distribution setup
params = {
    "criterion": ['gini', 'entropy'],
    "max_depth": [None, 5, 10, 15, 20],
    "min_samples_split": [2, 5, 10, 20],
    "min_samples_leaf": [1, 2, 5, 10],
    "max_features": ['None', 'sqrt', 'log2']

}

#  Randomized Search execution
rs = RandomizedSearchCV(
    estimator=model,
    param_distributions=params,
    n_iter=10,
    cv=skf,
    scoring='f1',
    n_jobs=-1,
    random_state=42
)

# Training
rs.fit(X_train_ad, y_train_ad)

# Best model extract
final_dt_model = rs.best_estimator_

print(f"Best Parameters: {rs.best_params_}")
print(f"Best CV Accuracy: {round(rs.best_score_, 4)}")

run_model_evaluation("Decision Tree", final_dt_model)

Best Parameters: {'min_samples_split': 5, 'min_samples_leaf': 2, 'max_features': 'log2', 'max_depth': None, 'criterion': 'entropy'}
Best CV Accuracy: 0.8667
--- For: Decision Tree ---


,Fold,Classifier,Accuracy,Precision,Recall,Specificity,G-Mean,F1-Score,MCC,Kappa,ROC-AUC,Balanced Accuracy,Train_Time
0,1,Decision Tree,0.8668,0.8952,0.8308,0.9029,0.8661,0.8618,0.7356,0.7337,0.9216,0.8668,0.0886
1,2,Decision Tree,0.8761,0.9025,0.8431,0.9090,0.8754,0.8718,0.7538,0.7522,0.9233,0.8761,0.0889
2,3,Decision Tree,0.8738,0.9014,0.8393,0.9083,0.8731,0.8692,0.7494,0.7477,0.9257,0.8738,0.0946
3,4,Decision Tree,0.8701,0.9066,0.8249,0.9152,0.8689,0.8638,0.7432,0.7401,0.9241,0.8700,0.0842
4,5,Decision Tree,0.8780,0.9085,0.8403,0.9155,0.8771,0.8731,0.7580,0.7559,0.9306,0.8779,0.0769
5,6,Decision Tree,0.8721,0.8989,0.8382,0.9060,0.8714,0.8675,0.7459,0.7442,0.9279,0.8721,0.0874
6,7,Decision Tree,0.8648,0.8863,0.8365,0.8930,0.8643,0.8607,0.7307,0.7295,0.9256,0.8647,0.0834
7,8,Decision Tree,0.8672,0.8935,0.8334,0.9008,0.8665,0.8624,0.7360,0.7343,0.9214,0.8671,0.1006
8,9,Decision Tree,0.8745,0.8963,0.8469,0.9022,0.8741,0.8709,0.7502,0.7490,0.9291,0.8745,0.0822
9,10,Decision Tree,0.8720,0.9101,0.8253,0.9186,0.8707,0.8656,0.7472,0.7439,0.9266,0.8719,0.0844


# **Random Forest**

In [17]:
from sklearn.ensemble import RandomForestClassifier
#  Model base initialize
model = RandomForestClassifier(random_state=42,n_jobs=-1)

# Parameter distribution setup
params = {
  "n_estimators": [100, 200],
  "max_depth": [10, 20, None],
  "min_samples_split": [2, 5],
  "max_features": ['sqrt']
}

#  Randomized Search execution
rs = RandomizedSearchCV(
    estimator=model,
    param_distributions=params,
    n_iter=5,
    cv=skf,
    scoring='f1',
    n_jobs=-1,
    random_state=42
)

# Training
rs.fit(X_train_ad, y_train_ad)

# Best model extract
final_rf_model = rs.best_estimator_

print(f"Best Parameters: {rs.best_params_}")
print(f"Best CV Accuracy: {round(rs.best_score_, 4)}")

run_model_evaluation("Random Forest", final_rf_model)

Best Parameters: {'n_estimators': 100, 'min_samples_split': 5, 'max_features': 'sqrt', 'max_depth': None}
Best CV Accuracy: 0.9419
--- For: Random Forest ---


,Fold,Classifier,Accuracy,Precision,Recall,Specificity,G-Mean,F1-Score,MCC,Kappa,ROC-AUC,Balanced Accuracy,Train_Time
0,1,Random Forest,0.9452,0.9548,0.9346,0.9559,0.9452,0.9446,0.8907,0.8905,0.9841,0.9452,4.7609
1,2,Random Forest,0.9435,0.9531,0.9329,0.9542,0.9435,0.9429,0.8872,0.8870,0.9831,0.9435,5.9162
2,3,Random Forest,0.9439,0.9497,0.9373,0.9504,0.9438,0.9434,0.8878,0.8877,0.9839,0.9438,8.4760
3,4,Random Forest,0.9421,0.9542,0.9287,0.9555,0.9420,0.9413,0.8846,0.8843,0.9819,0.9421,8.7237
4,5,Random Forest,0.9397,0.9464,0.9321,0.9473,0.9397,0.9392,0.8796,0.8795,0.9833,0.9397,5.9830
5,6,Random Forest,0.9440,0.9541,0.9328,0.9552,0.9439,0.9433,0.8883,0.8880,0.9842,0.9440,12.1887
6,7,Random Forest,0.9459,0.9562,0.9345,0.9573,0.9458,0.9452,0.8920,0.8918,0.9855,0.9459,11.4103
7,8,Random Forest,0.9331,0.9389,0.9263,0.9398,0.9330,0.9326,0.8662,0.8661,0.9818,0.9331,9.0298
8,9,Random Forest,0.9437,0.9541,0.9322,0.9552,0.9436,0.9430,0.8876,0.8874,0.9839,0.9437,4.9644
9,10,Random Forest,0.9444,0.9519,0.9359,0.9528,0.9443,0.9439,0.8889,0.8887,0.9850,0.9444,6.2290


# **GBM**

In [18]:
from sklearn.ensemble import GradientBoostingClassifier
#  Model base initialize
model = GradientBoostingClassifier(random_state=42)

# Parameter distribution setup
params = {
  "n_estimators": [100, 200],
  "learning_rate": [0.1, 0.2],
  "max_depth": [3, 5],
  "subsample": [0.8],
  "max_features": ['sqrt']
}

#  Randomized Search execution
rs = RandomizedSearchCV(
    estimator=model,
    param_distributions=params,
    n_iter=10,
    cv=skf,
    scoring='f1',
    n_jobs=-1,
    random_state=42
)

# Training
rs.fit(X_train_ad, y_train_ad)

# Best model extract
final_gbm_model = rs.best_estimator_

print(f"Best Parameters: {rs.best_params_}")
print(f"Best CV Accuracy: {round(rs.best_score_, 4)}")

run_model_evaluation("GBM", final_gbm_model)

Best Parameters: {'subsample': 0.8, 'n_estimators': 200, 'max_features': 'sqrt', 'max_depth': 5, 'learning_rate': 0.2}
Best CV Accuracy: 0.931
--- For: GBM ---


,Fold,Classifier,Accuracy,Precision,Recall,Specificity,G-Mean,F1-Score,MCC,Kappa,ROC-AUC,Balanced Accuracy,Train_Time
0,1,GBM,0.9331,0.9713,0.8924,0.9737,0.9322,0.9302,0.8690,0.8662,0.9728,0.9330,12.9645
1,2,GBM,0.9343,0.9703,0.8959,0.9726,0.9335,0.9316,0.8711,0.8686,0.9731,0.9342,12.6657
2,3,GBM,0.9326,0.9606,0.9020,0.9631,0.9320,0.9304,0.8667,0.8651,0.9724,0.9325,12.4355
3,4,GBM,0.9273,0.9608,0.8907,0.9637,0.9265,0.9244,0.8568,0.8545,0.9705,0.9272,10.8017
4,5,GBM,0.9350,0.9625,0.9051,0.9648,0.9344,0.9329,0.8714,0.8699,0.9741,0.9349,12.2376
5,6,GBM,0.9367,0.9719,0.8992,0.9740,0.9359,0.9341,0.8758,0.8733,0.9754,0.9366,14.9370
6,7,GBM,0.9336,0.9664,0.8982,0.9689,0.9329,0.9311,0.8693,0.8672,0.9768,0.9335,10.8529
7,8,GBM,0.9291,0.9593,0.8962,0.9620,0.9285,0.9266,0.8601,0.8583,0.9723,0.9291,12.8123
8,9,GBM,0.9350,0.9693,0.8983,0.9716,0.9342,0.9324,0.8722,0.8699,0.9752,0.9349,9.5523
9,10,GBM,0.9384,0.9682,0.9065,0.9702,0.9378,0.9363,0.8785,0.8767,0.9762,0.9384,7.4577


# **XGboost**

In [19]:
from xgboost import XGBClassifier
#  Model base initialize
model = XGBClassifier(eval_metric='logloss', use_label_encoder=False, random_state=42, tree_method='hist')

# Parameter distribution setup
params = {
"n_estimators": [50, 100],
"learning_rate": [0.1, 0.2],
"max_depth": [3, 5, 6],
"subsample": [0.8],
"colsample_bytree":[0.8]
}

#  Randomized Search execution
rs = RandomizedSearchCV(
    estimator=model,
    param_distributions=params,
    n_iter=10,
    cv=skf,
    scoring='f1',
    n_jobs=-1,
    random_state=42
)

# Training
rs.fit(X_train_ad, y_train_ad)

# Best model extract
final_xg_model = rs.best_estimator_

print(f"Best Parameters: {rs.best_params_}")
print(f"Best CV Accuracy: {round(rs.best_score_, 4)}")

run_model_evaluation("xgboost", final_xg_model)

Best Parameters: {'subsample': 0.8, 'n_estimators': 100, 'max_depth': 6, 'learning_rate': 0.2, 'colsample_bytree': 0.8}
Best CV Accuracy: 0.9328
--- For: xgboost ---


,Fold,Classifier,Accuracy,Precision,Recall,Specificity,G-Mean,F1-Score,MCC,Kappa,ROC-AUC,Balanced Accuracy,Train_Time
0,1,xgboost,0.9329,0.9741,0.8893,0.9764,0.9319,0.9298,0.8691,0.8658,0.9743,0.9329,1.3272
1,2,xgboost,0.9356,0.9725,0.8965,0.9747,0.9348,0.9330,0.8740,0.8713,0.9719,0.9356,2.5484
2,3,xgboost,0.9363,0.9649,0.9054,0.9672,0.9358,0.9342,0.8743,0.8726,0.9730,0.9363,1.7603
3,4,xgboost,0.9302,0.9665,0.8910,0.9692,0.9293,0.9272,0.8629,0.8603,0.9700,0.9301,1.2811
4,5,xgboost,0.9348,0.9641,0.9030,0.9665,0.9342,0.9326,0.8713,0.8696,0.9742,0.9348,3.1417
5,6,xgboost,0.9385,0.9744,0.9006,0.9764,0.9377,0.9361,0.8796,0.8771,0.9764,0.9385,2.4577
6,7,xgboost,0.9360,0.9718,0.8979,0.9740,0.9352,0.9334,0.8745,0.8720,0.9771,0.9359,2.7316
7,8,xgboost,0.9320,0.9660,0.8955,0.9685,0.9313,0.9294,0.8664,0.8641,0.9714,0.9320,3.8030
8,9,xgboost,0.9387,0.9762,0.8993,0.9781,0.9379,0.9362,0.8802,0.8774,0.9750,0.9387,3.8199
9,10,xgboost,0.9387,0.9699,0.9054,0.9719,0.9381,0.9366,0.8794,0.8774,0.9752,0.9387,2.6643


# **LightGBM**

In [20]:
from lightgbm import LGBMClassifier
#  Model base initialize
model = LGBMClassifier(random_state=42,verbosity=-1)

# Parameter distribution setup
params = {
"n_estimators": [100, 150],
"learning_rate": [0.05, 0.1],
"num_leaves": [31, 40],
"boosting_type": ['gbdt']
}

#  Randomized Search execution
rs = RandomizedSearchCV(
    estimator=model,
    param_distributions=params,
    n_iter=10,
    cv=skf,
    scoring='f1',
    n_jobs=-1,
    random_state=42
)

# Training
rs.fit(X_train_ad, y_train_ad)

# Best model extract
final_lgbm_model = rs.best_estimator_

print(f"Best Parameters: {rs.best_params_}")
print(f"Best CV Accuracy: {round(rs.best_score_, 4)}")

run_model_evaluation("LightGBM", final_lgbm_model)

Best Parameters: {'num_leaves': 40, 'n_estimators': 150, 'learning_rate': 0.1, 'boosting_type': 'gbdt'}
Best CV Accuracy: 0.9358
--- For: LightGBM ---


,Fold,Classifier,Accuracy,Precision,Recall,Specificity,G-Mean,F1-Score,MCC,Kappa,ROC-AUC,Balanced Accuracy,Train_Time
0,1,LightGBM,0.9358,0.9729,0.8965,0.9750,0.9350,0.9331,0.8743,0.8716,0.9757,0.9358,1.6767
1,2,LightGBM,0.9372,0.9705,0.9017,0.9726,0.9365,0.9348,0.8766,0.8744,0.9737,0.9372,1.6754
2,3,LightGBM,0.9363,0.9643,0.9061,0.9665,0.9358,0.9343,0.8742,0.8726,0.9746,0.9363,1.6428
3,4,LightGBM,0.9348,0.9703,0.8968,0.9726,0.9340,0.9321,0.8721,0.8696,0.9717,0.9347,1.6354
4,5,LightGBM,0.9389,0.9661,0.9095,0.9682,0.9384,0.9370,0.8793,0.8778,0.9755,0.9389,1.6055
5,6,LightGBM,0.9416,0.9753,0.9061,0.9771,0.9409,0.9394,0.8855,0.8833,0.9770,0.9416,2.0996
6,7,LightGBM,0.9389,0.9723,0.9034,0.9744,0.9382,0.9366,0.8800,0.8778,0.9781,0.9389,2.0146
7,8,LightGBM,0.9351,0.9676,0.9003,0.9699,0.9344,0.9327,0.8724,0.8702,0.9738,0.9351,1.6484
8,9,LightGBM,0.9401,0.9763,0.9020,0.9781,0.9393,0.9377,0.8827,0.8802,0.9763,0.9401,1.6611
9,10,LightGBM,0.9418,0.9736,0.9082,0.9754,0.9412,0.9397,0.8856,0.8836,0.9777,0.9418,1.6537


# **Catboost**

In [21]:
from catboost import CatBoostClassifier
#  Model base initialize
model = CatBoostClassifier(random_state=42,verbose=0, thread_count=-1)

# Parameter distribution setup
params = {
"iterations": [100, 200],
"learning_rate": [0.05, 0.1],
"depth": [4, 6]
}

#  Randomized Search execution
rs = RandomizedSearchCV(
    estimator=model,
    param_distributions=params,
    n_iter=10,
    cv=skf,
    scoring='f1',
    n_jobs=-1,
    random_state=42
)

# Training
rs.fit(X_train_ad, y_train_ad)

# Best model extract
final_cat_model = rs.best_estimator_

print(f"Best Parameters: {rs.best_params_}")
print(f"Best CV Accuracy: {round(rs.best_score_, 4)}")

run_model_evaluation("Catboost", final_cat_model)

Best Parameters: {'learning_rate': 0.1, 'iterations': 200, 'depth': 6}
Best CV Accuracy: 0.9313
--- For: Catboost ---


,Fold,Classifier,Accuracy,Precision,Recall,Specificity,G-Mean,F1-Score,MCC,Kappa,ROC-AUC,Balanced Accuracy,Train_Time
0,1,Catboost,0.9302,0.9787,0.8794,0.9808,0.9287,0.9264,0.8648,0.8603,0.9728,0.9301,6.0551
1,2,Catboost,0.9351,0.9771,0.8911,0.9791,0.9341,0.9321,0.8736,0.8703,0.9724,0.9351,6.5199
2,3,Catboost,0.9344,0.9693,0.8972,0.9716,0.9337,0.9318,0.8713,0.8689,0.9719,0.9344,5.0498
3,4,Catboost,0.9315,0.9747,0.8859,0.9771,0.9304,0.9282,0.8666,0.8630,0.9694,0.9315,6.3599
4,5,Catboost,0.9351,0.9704,0.8975,0.9726,0.9343,0.9325,0.8727,0.8702,0.9742,0.9351,4.9261
5,6,Catboost,0.9344,0.9742,0.8924,0.9764,0.9335,0.9315,0.8719,0.8689,0.9755,0.9344,5.1016
6,7,Catboost,0.9363,0.9729,0.8975,0.9750,0.9355,0.9337,0.8753,0.8726,0.9773,0.9363,5.8426
7,8,Catboost,0.9303,0.9693,0.8886,0.9720,0.9294,0.9272,0.8636,0.8607,0.9732,0.9303,4.8961
8,9,Catboost,0.9363,0.9764,0.8941,0.9784,0.9353,0.9335,0.8758,0.8726,0.9737,0.9363,6.0627
9,10,Catboost,0.9385,0.9727,0.9024,0.9747,0.9378,0.9362,0.8794,0.8771,0.9748,0.9385,4.9318


# **SVM**

In [24]:
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
#  Model base initialize
base_model = LinearSVC(random_state=42, max_iter=2000, dual=False)
model = CalibratedClassifierCV(base_model)

# Parameter distribution setup
params = {
"estimator__C": [0.1, 1, 10]
}

#  Randomized Search execution
rs = RandomizedSearchCV(
    estimator=model,
    param_distributions=params,
    n_iter=3,
    cv=skf,
    scoring='f1',
    n_jobs=-1,
    random_state=42
)

# Training
rs.fit(X_train_ad, y_train_ad)

# Best model extract
final_svc_model = rs.best_estimator_

print(f"Best Parameters: {rs.best_params_}")
print(f"Best CV Accuracy: {round(rs.best_score_, 4)}")

run_model_evaluation("SVM", final_svc_model)

Best Parameters: {'estimator__C': 0.1}
Best CV Accuracy: 0.6535
--- For: SVM ---


,Fold,Classifier,Accuracy,Precision,Recall,Specificity,G-Mean,F1-Score,MCC,Kappa,ROC-AUC,Balanced Accuracy,Train_Time
0,1,SVM,0.6964,0.7446,0.5971,0.7955,0.6892,0.6627,0.4006,0.3927,0.7524,0.6963,6.6659
1,2,SVM,0.6925,0.7361,0.5992,0.7856,0.6861,0.6606,0.3916,0.3848,0.7505,0.6924,5.3329
2,3,SVM,0.6780,0.7196,0.5822,0.7736,0.6711,0.6437,0.3626,0.3559,0.7386,0.6779,5.7468
3,4,SVM,0.6854,0.7330,0.5822,0.7883,0.6775,0.6490,0.3787,0.3706,0.7449,0.6853,5.9524
4,5,SVM,0.6840,0.7279,0.5867,0.7811,0.6770,0.6497,0.3750,0.3679,0.7449,0.6839,5.3846
5,6,SVM,0.6898,0.7331,0.5960,0.7835,0.6833,0.6575,0.3864,0.3795,0.7489,0.6897,6.2312
6,7,SVM,0.6962,0.7462,0.5936,0.7986,0.6885,0.6612,0.4007,0.3922,0.7555,0.6961,5.1878
7,8,SVM,0.6777,0.7203,0.5798,0.7753,0.6705,0.6425,0.3622,0.3552,0.7372,0.6776,6.3526
8,9,SVM,0.6955,0.7434,0.5964,0.7944,0.6883,0.6619,0.3987,0.3909,0.7461,0.6954,5.9776
9,10,SVM,0.6838,0.7322,0.5790,0.7886,0.6757,0.6466,0.3759,0.3676,0.7445,0.6838,6.4459


# **KNN**

In [25]:
from sklearn.neighbors import KNeighborsClassifier
#  Model base initialize
model = KNeighborsClassifier()

# Parameter distribution setup
params = {
"n_neighbors": [3, 5, 7],
"weights": ['uniform', 'distance'],
}

#  Randomized Search execution
rs = RandomizedSearchCV(
    estimator=model,
    param_distributions=params,
    n_iter=5,
    cv=skf,
    scoring='f1',
    n_jobs=-1,
    random_state=42
)

# Training
rs.fit(X_train_ad, y_train_ad)

# Best model extract
final_knn_model = rs.best_estimator_

print(f"Best Parameters: {rs.best_params_}")
print(f"Best CV Accuracy: {round(rs.best_score_, 4)}")

run_model_evaluation("KNN", final_knn_model)

Best Parameters: {'weights': 'distance', 'n_neighbors': 3}
Best CV Accuracy: 0.9075
--- For: KNN ---


,Fold,Classifier,Accuracy,Precision,Recall,Specificity,G-Mean,F1-Score,MCC,Kappa,ROC-AUC,Balanced Accuracy,Train_Time
0,1,KNN,0.9012,0.8553,0.9657,0.8369,0.8990,0.9072,0.8093,0.8025,0.9412,0.9013,0.0049
1,2,KNN,0.9069,0.8589,0.9736,0.8403,0.9045,0.9127,0.8212,0.8138,0.9451,0.9070,0.0036
2,3,KNN,0.8999,0.8510,0.9692,0.8307,0.8973,0.9063,0.8076,0.7998,0.9384,0.8999,0.0047
3,4,KNN,0.9014,0.8521,0.9712,0.8317,0.8988,0.9078,0.8108,0.8028,0.9388,0.9015,0.0037
4,5,KNN,0.8988,0.8468,0.9736,0.8242,0.8958,0.9058,0.8068,0.7977,0.9414,0.8989,0.0152
5,6,KNN,0.9060,0.8586,0.9719,0.8403,0.9037,0.9118,0.8192,0.8121,0.9423,0.9061,0.0039
6,7,KNN,0.9038,0.8583,0.9671,0.8406,0.9017,0.9094,0.8142,0.8076,0.9398,0.9039,0.0037
7,8,KNN,0.8937,0.8440,0.9657,0.8218,0.8909,0.9008,0.7957,0.7874,0.9330,0.8938,0.0042
8,9,KNN,0.8971,0.8460,0.9709,0.8235,0.8941,0.9041,0.8031,0.7943,0.9362,0.8972,0.0037
9,10,KNN,0.9031,0.8551,0.9705,0.8358,0.9006,0.9092,0.8137,0.8062,0.9394,0.9032,0.0053


# **MLP**

In [27]:
from sklearn.neural_network import MLPClassifier
#  Model base initialize
model = MLPClassifier(max_iter=200, random_state=42, early_stopping=True)

# Parameter distribution setup
params = {
"hidden_layer_sizes": [(64,),(64,32)],
            "alpha": [0.0001, 0.01]
}

#  Randomized Search execution
rs = RandomizedSearchCV(
    estimator=model,
    param_distributions=params,
    n_iter=5,
    cv=skf,
    scoring='f1',
    n_jobs=-1,
    random_state=42
)

# Training
rs.fit(X_train_ad, y_train_ad)

# Best model extract
final_mlp_model = rs.best_estimator_

print(f"Best Parameters: {rs.best_params_}")
print(f"Best CV Accuracy: {round(rs.best_score_, 4)}")

run_model_evaluation("MLP", final_mlp_model)

Best Parameters: {'hidden_layer_sizes': (64, 32), 'alpha': 0.01}
Best CV Accuracy: 0.8744
--- For: MLP ---


,Fold,Classifier,Accuracy,Precision,Recall,Specificity,G-Mean,F1-Score,MCC,Kappa,ROC-AUC,Balanced Accuracy,Train_Time
0,1,MLP,0.8781,0.8605,0.9024,0.8540,0.8778,0.8809,0.7572,0.7563,0.9451,0.8782,26.5692
1,2,MLP,0.8747,0.8656,0.8869,0.8625,0.8746,0.8761,0.7497,0.7494,0.9445,0.8747,27.1927
2,3,MLP,0.8786,0.8566,0.9092,0.8482,0.8781,0.8821,0.7587,0.7573,0.9416,0.8787,27.7182
3,4,MLP,0.8711,0.8600,0.8862,0.8560,0.8710,0.8729,0.7426,0.7422,0.9427,0.8711,25.0094
4,5,MLP,0.8685,0.8651,0.8729,0.8642,0.8685,0.8690,0.7371,0.7371,0.9399,0.8685,24.1557
5,6,MLP,0.8692,0.8631,0.8773,0.8611,0.8692,0.8702,0.7385,0.7384,0.9401,0.8692,20.2480
6,7,MLP,0.8720,0.8671,0.8783,0.8656,0.8719,0.8727,0.7440,0.7439,0.9430,0.8720,19.8391
7,8,MLP,0.8773,0.8729,0.8828,0.8718,0.8773,0.8778,0.7546,0.7545,0.9439,0.8773,36.3271
8,9,MLP,0.8687,0.8491,0.8965,0.8409,0.8683,0.8722,0.7386,0.7374,0.9398,0.8687,17.5598
9,10,MLP,0.8684,0.8595,0.8804,0.8563,0.8683,0.8699,0.7370,0.7367,0.9381,0.8684,18.5087


# **Bagging**

In [28]:
from sklearn.ensemble import BaggingClassifier
#  Model base initialize
model = BaggingClassifier(random_state=42)

# Parameter distribution setup
params = {
"n_estimators": [10, 50],
"max_samples": [0.5, 1.0],
"max_features": [0.5, 1.0],
}

#  Randomized Search execution
rs = RandomizedSearchCV(
    estimator=model,
    param_distributions=params,
    n_iter=5,
    cv=skf,
    scoring='f1',
    n_jobs=-1,
    random_state=42
)

# Training
rs.fit(X_train_ad, y_train_ad)

# Best model extract
final_bg_model = rs.best_estimator_

print(f"Best Parameters: {rs.best_params_}")
print(f"Best CV Accuracy: {round(rs.best_score_, 4)}")

run_model_evaluation("Bagging", final_bg_model)

Best Parameters: {'n_estimators': 50, 'max_samples': 0.5, 'max_features': 0.5}
Best CV Accuracy: 0.9373
--- For: Bagging ---


,Fold,Classifier,Accuracy,Precision,Recall,Specificity,G-Mean,F1-Score,MCC,Kappa,ROC-AUC,Balanced Accuracy,Train_Time
0,1,Bagging,0.9386,0.9665,0.9085,0.9685,0.9381,0.9366,0.8787,0.8771,0.9796,0.9385,4.8634
1,2,Bagging,0.9379,0.9624,0.9113,0.9644,0.9375,0.9361,0.8770,0.8757,0.9776,0.9379,5.7932
2,3,Bagging,0.9382,0.9568,0.9178,0.9586,0.9380,0.9369,0.8771,0.8764,0.9792,0.9382,4.7861
3,4,Bagging,0.9372,0.9610,0.9112,0.9631,0.9368,0.9354,0.8755,0.8744,0.9764,0.9372,5.6515
4,5,Bagging,0.9404,0.9602,0.9188,0.9620,0.9402,0.9391,0.8817,0.8809,0.9787,0.9404,4.7931
5,6,Bagging,0.9397,0.9649,0.9126,0.9668,0.9393,0.9380,0.8808,0.8795,0.9797,0.9397,4.8324
6,7,Bagging,0.9420,0.9650,0.9171,0.9668,0.9416,0.9404,0.8850,0.8839,0.9820,0.9419,5.7059
7,8,Bagging,0.9320,0.9533,0.9085,0.9555,0.9317,0.9303,0.8650,0.8641,0.9751,0.9320,4.7756
8,9,Bagging,0.9404,0.9629,0.9161,0.9648,0.9401,0.9389,0.8819,0.8809,0.9799,0.9404,5.6942
9,10,Bagging,0.9427,0.9598,0.9239,0.9613,0.9425,0.9415,0.8859,0.8853,0.9805,0.9426,4.8069


# **Stacking**

In [30]:
from sklearn.ensemble import StackingClassifier
from sklearn.linear_model import LogisticRegression
base_learners = [
    ('rf', final_rf_model),
    ('xgb', final_xg_model),
    ('lgbm', final_lgbm_model)
]

meta_model = LogisticRegression()

stack_model = StackingClassifier(
    estimators=base_learners,
    final_estimator=meta_model,
    cv=3,
    n_jobs=-1
)
stack_model.fit(X_train_ad, y_train_ad)

run_model_evaluation("Stacking Classifier", stack_model)

--- For: Stacking Classifier ---


,Fold,Classifier,Accuracy,Precision,Recall,Specificity,G-Mean,F1-Score,MCC,Kappa,ROC-AUC,Balanced Accuracy,Train_Time
0,1,Stacking Classifier,0.9442,0.9451,0.9431,0.9453,0.9442,0.9441,0.8884,0.8884,0.9848,0.9442,31.8456
1,2,Stacking Classifier,0.9403,0.9419,0.9383,0.9422,0.9403,0.9401,0.8805,0.8805,0.9837,0.9403,31.9386
2,3,Stacking Classifier,0.9396,0.9358,0.9438,0.9354,0.9396,0.9398,0.8792,0.8792,0.9846,0.9396,31.9670
3,4,Stacking Classifier,0.9397,0.9433,0.9356,0.9439,0.9397,0.9394,0.8795,0.8795,0.9822,0.9397,33.8220
4,5,Stacking Classifier,0.9384,0.9350,0.9421,0.9347,0.9384,0.9385,0.8768,0.8768,0.9839,0.9384,32.4957
5,6,Stacking Classifier,0.9440,0.9444,0.9435,0.9446,0.9440,0.9439,0.8881,0.8881,0.9852,0.9440,34.1026
6,7,Stacking Classifier,0.9430,0.9437,0.9421,0.9439,0.9430,0.9429,0.8860,0.8860,0.9862,0.9430,29.9339
7,8,Stacking Classifier,0.9329,0.9302,0.9359,0.9299,0.9329,0.9330,0.8658,0.8658,0.9825,0.9329,29.1607
8,9,Stacking Classifier,0.9432,0.9443,0.9418,0.9446,0.9432,0.9431,0.8863,0.8863,0.9846,0.9432,31.8226
9,10,Stacking Classifier,0.9432,0.9401,0.9466,0.9398,0.9432,0.9433,0.8864,0.8863,0.9857,0.9432,33.4648


In [33]:
!pip install xlsxwriter

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 175.3/175.3 kB 6.8 MB/s eta 0:00:00


In [39]:
import pandas as pd

def save_to_final_excel_v3(results, filename="RESULT_KFOLD_FORMAT_FINAL.xlsx"):
    if not results:

        return


    df = pd.DataFrame(results)


    df = df.drop_duplicates(subset=['Fold', 'Classifier'], keep='first')


    cols_map = {c.lower(): c for c in df.columns}
    required_metrics = ['accuracy', 'precision', 'recall', 'specificity', 'f1-score', 'g-mean', 'roc-auc', 'mcc', 'kappa']

    actual_cols = []
    for m in required_metrics:
        if m in cols_map:
            actual_cols.append(cols_map[m])
        elif m.replace('-', '_') in cols_map:
            actual_cols.append(cols_map[m.replace('-', '_')])

    with pd.ExcelWriter(filename, engine='xlsxwriter') as writer:

        for model_name in df['Classifier'].unique():
            model_df = df[df['Classifier'] == model_name].copy()


            model_df['Fold'] = pd.to_numeric(model_df['Fold'], errors='coerce')
            model_df = model_df.sort_values(by='Fold')


            final_column_order = ['Fold', 'Classifier'] + [c for c in actual_cols if c in model_df.columns]


            sheet_name = str(model_name)[:31].replace("/", "_")
            model_df[final_column_order].to_excel(writer, sheet_name=sheet_name, index=False)




save_to_final_excel_v3(results)